# Fake News Evaluation — GitHub + Google Fact Check

This notebook uses your GitHub repository directly:

`https://github.com/KP-365/Fake_news`

Run every cell from top to bottom. It will:

1. Clone the repository.
2. Pull any Git LFS files.
3. locate the project files automatically;
4. install dependencies;
5. request your Google Fact Check API key securely;
6. locate or upload missing trained-model files;
7. test the classifier and verifier;
8. run `evaluation/eval_escalation.py`;
9. locate, display, and download the generated results.

> In Colab, select **Runtime → Change runtime type → T4 GPU** before starting.


## Part 1 — Check the Colab runtime

In [1]:
import os
import sys
from pathlib import Path

print("Python:", sys.version)
print("Current working directory:", os.getcwd())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("PyTorch check failed:", exc)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Current working directory: /content
PyTorch: 2.11.0+cpu
CUDA available: False


## Part 2 — Clone the GitHub repository

In [2]:
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/content")

REPO_URL = "https://github.com/KP-365/Fake_news.git"
REPO_DIR = Path("/content/Fake_news")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    check=True
)

project_root = REPO_DIR
os.chdir(project_root)

print("Repository cloned to:", project_root)
print("\nTop-level contents:")
for item in sorted(project_root.iterdir()):
    print(" -", item.name)


Repository cloned to: /content/Fake_news

Top-level contents:
 - .env.example
 - .git
 - .gitattributes
 - .gitignore
 - DESIGN.md
 - README.md
 - TASKS.md
 - app.py
 - docs
 - eval_FakeNews.ipynb
 - eval_MCFakeNews.ipynb
 - eval_escalation.ipynb
 - eval_faithfulness.ipynb
 - evaluation
 - explain.py
 - google factchek
 - models
 - pipeline.py
 - predict.py
 - requirements.txt
 - scaffold_FakeNews_finn's_training.ipynb
 - scripts
 - space
 - tests
 - verify.py


## Part 3 — Pull Git LFS files and inspect the repository

In [3]:
import os
import subprocess
from pathlib import Path

os.chdir(project_root)

subprocess.run(["git", "lfs", "install"], check=False)
subprocess.run(["git", "lfs", "pull"], check=False)

print("\nImportant project files:")
for filename in [
    "predict.py",
    "verify.py",
    "requirements.txt",
    "eval_escalation.py",
    "classifier_head.pt",
    "adapter_model.safetensors",
    "adapter_config.json",
]:
    matches = list(project_root.rglob(filename))
    if matches:
        for match in matches:
            print(f"FOUND {filename}: {match}")
    else:
        print(f"MISSING {filename}")



Important project files:
FOUND predict.py: /content/Fake_news/predict.py
FOUND predict.py: /content/Fake_news/google factchek/predict.py
FOUND verify.py: /content/Fake_news/verify.py
FOUND verify.py: /content/Fake_news/google factchek/verify.py
FOUND requirements.txt: /content/Fake_news/requirements.txt
FOUND requirements.txt: /content/Fake_news/google factchek/requirements.txt
FOUND eval_escalation.py: /content/Fake_news/evaluation/eval_escalation.py
FOUND eval_escalation.py: /content/Fake_news/google factchek/evaluation/eval_escalation.py
FOUND classifier_head.pt: /content/Fake_news/models/roberta-trained-welfake/classifier_head.pt
FOUND classifier_head.pt: /content/Fake_news/google factchek/models/roberta-trained-welfake/classifier_head.pt
FOUND adapter_model.safetensors: /content/Fake_news/models/roberta-trained-welfake/adapter_model.safetensors
FOUND adapter_model.safetensors: /content/Fake_news/google factchek/models/roberta-trained-welfake/adapter_model.safetensors
FOUND adapte

## Part 4 — Locate the runnable project root

In [5]:
from pathlib import Path
import os

candidates = []

for folder in [project_root] + [p for p in project_root.rglob("*") if p.is_dir()]:
    if (
        (folder / "predict.py").exists()
        and (folder / "verify.py").exists()
        and (folder / "evaluation" / "eval_escalation.py").exists()
    ):
        candidates.append(folder)

if not candidates:
    raise FileNotFoundError(
        "Could not find a folder containing predict.py, verify.py, "
        "and evaluation/eval_escalation.py."
    )

project_root = sorted(candidates, key=lambda p: len(p.parts))[0]
os.chdir(project_root)

print("Runnable project root:", project_root)
print("Current working directory:", os.getcwd())


Runnable project root: /content/Fake_news
Current working directory: /content/Fake_news


## Part 5 — Install dependencies

In [6]:
import os
from pathlib import Path

os.chdir(project_root)

requirements = project_root / "requirements.txt"

!python -m pip install --upgrade pip -q

if requirements.exists():
    !pip install -r "$requirements"
else:
    print("No requirements.txt found. Installing common dependencies.")
    !pip install torch transformers peft accelerate pandas numpy scikit-learn requests python-dotenv duckduckgo-search kagglehub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 111.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 21.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 128.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 94.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 104.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.6 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully unin

## Part 6 — Enter the Google Fact Check API key

The key is hidden while you type and is stored only for this Colab session.


In [7]:
import os
from getpass import getpass

api_key = getpass("Paste GOOGLE_FACT_CHECK_API_KEY: ").strip()

if not api_key:
    raise ValueError("No Google Fact Check API key was entered.")

os.environ["GOOGLE_FACT_CHECK_API_KEY"] = api_key
print("API key stored for this Colab session.")


Paste GOOGLE_FACT_CHECK_API_KEY: ··········
API key stored for this Colab session.


## Part 7 — Prepare the trained-model directory

In [8]:
import shutil
from pathlib import Path

expected_model_dir = project_root / "models" / "roberta-trained-welfake"
expected_model_dir.mkdir(parents=True, exist_ok=True)

required_model_files = [
    "classifier_head.pt",
    "adapter_model.safetensors",
    "adapter_config.json",
]

print("Expected model directory:", expected_model_dir)

# Search the whole cloned repository and copy files into the path expected by predict.py.
for filename in required_model_files:
    destination = expected_model_dir / filename

    if destination.exists():
        print("READY:", destination)
        continue

    matches = [
        p for p in project_root.rglob(filename)
        if p.resolve() != destination.resolve()
    ]

    if matches:
        shutil.copy2(matches[0], destination)
        print(f"Copied {matches[0]} -> {destination}")
    else:
        print("NOT FOUND:", filename)

print("\nCurrent model directory contents:")
for item in sorted(expected_model_dir.iterdir()):
    print(" -", item.name)

missing_model_files = [
    filename
    for filename in required_model_files
    if not (expected_model_dir / filename).exists()
]

print("\nMissing model files:", missing_model_files or "None")


Expected model directory: /content/Fake_news/models/roberta-trained-welfake
READY: /content/Fake_news/models/roberta-trained-welfake/classifier_head.pt
READY: /content/Fake_news/models/roberta-trained-welfake/adapter_model.safetensors
READY: /content/Fake_news/models/roberta-trained-welfake/adapter_config.json

Current model directory contents:
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - classifier_head.pt
 - tokenizer.json
 - tokenizer_config.json

Missing model files: None


## Part 8 — Upload missing model files only if required

Run this cell only when Part 7 lists missing files. Select files such as:

- `classifier_head.pt`
- `adapter_model.safetensors`
- `adapter_config.json`


In [9]:
# Run only if Part 7 reported missing model files.

if missing_model_files:
    from google.colab import files

    print("Please upload:", missing_model_files)
    uploaded_models = files.upload()

    for filename, data in uploaded_models.items():
        destination = expected_model_dir / Path(filename).name
        destination.write_bytes(data)
        print("Saved:", destination)

    missing_model_files = [
        filename
        for filename in required_model_files
        if not (expected_model_dir / filename).exists()
    ]

    if missing_model_files:
        raise FileNotFoundError(
            "Still missing required model files: "
            + ", ".join(missing_model_files)
        )

    print("All required model files are now present.")
else:
    print("No upload is necessary. All required files are already present.")


No upload is necessary. All required files are already present.


## Part 9 — Validate model files before running

In [10]:
from pathlib import Path

for filename in required_model_files:
    path = expected_model_dir / filename
    if not path.exists():
        raise FileNotFoundError(f"Required model file is missing: {path}")

    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{filename}: {size_mb:.2f} MB")

    # Git LFS pointer files are tiny text files rather than real model weights.
    if path.stat().st_size < 1024:
        preview = path.read_text(errors="ignore")[:200]
        if "git-lfs" in preview:
            raise RuntimeError(
                f"{filename} is a Git LFS pointer rather than the real file. "
                "The real model file must be uploaded."
            )

print("Model-file validation passed.")


classifier_head.pt: 2.26 MB
adapter_model.safetensors: 1.13 MB
adapter_config.json: 0.00 MB
Model-file validation passed.


## Part 10 — Test the Google Fact Check verifier

In [11]:
import os
os.chdir(project_root)

!python verify.py "The Earth is flat"


Retrieving evidence...
Retrieved 5 evidence result(s).
config.json: 100% 1.09k/1.09k [00:00<00:00, 2.71MB/s]
tokenizer_config.json: 100% 1.28k/1.28k [00:00<00:00, 4.15MB/s]

spm.model: downloading bytes:   8% 188k/2.46M [00:01<00:19, 115kB/s]
spm.model: downloading bytes: 100% 1.64M/1.64M [00:01<00:00, 944kB/s,  158kB/s  ]
spm.model: reconstructing file: 100% 2.46M/2.46M [00:01<00:00, 1.42MB/s,  237kB/s  ]
tokenizer.json: 100% 8.66M/8.66M [00:00<00:00, 116MB/s]
added_tokens.json: 100% 23.0/23.0 [00:00<00:00, 106kB/s]
special_tokens_map.json: 100% 286/286 [00:00<00:00, 1.27MB/s]

model.safetensors: downloading bytes:  58% 215M/369M [00:02<00:00, 174MB/s, 17.6MB/s  ]
model.safetensors: downloading bytes:  98% 363M/369M [00:03<00:00, 160MB/s, 29.6MB/s  ]
model.safetensors: reconstructing file:  66% 242M/369M [00:04<00:01, 67.2MB/s, 6.24MB/s  ]
model.safetensors: reconstructing file:  88% 326M/369M [00:04<00:00, 99.1MB/s, 21.4MB/s  ]
model.safetensors: downloading bytes: 100% 369M/369M [00

## Part 11 — Test the RoBERTa classifier

In [12]:
import os
os.chdir(project_root)

!python predict.py "Scientists have confirmed that the Earth orbits the Sun."


config.json: 100% 481/481 [00:00<00:00, 987kB/s]

model.safetensors: downloading bytes:  45% 225M/499M [00:03<00:01, 155MB/s, 17.9MB/s  ]
model.safetensors: downloading bytes:  61% 303M/499M [00:03<00:02, 94.5MB/s, 24.9MB/s  ]
model.safetensors: downloading bytes:  64% 321M/499M [00:04<00:01, 92.9MB/s, 26.1MB/s  ]
model.safetensors: downloading bytes:  67% 333M/499M [00:04<00:01, 83.7MB/s, 26.6MB/s  ]
model.safetensors: downloading bytes: 100% 335M/335M [00:04<00:00, 76.3MB/s, 27.2MB/s  ]
model.safetensors: reconstructing file: 100% 499M/499M [00:04<00:00, 114MB/s, 44.2MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 5115.99it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head

## Part 12 — Run the complete evaluation

This is the cell that should create the results CSV and summary JSON.


In [13]:
import os
import subprocess

os.chdir(project_root)

completed = subprocess.run(
    ["python", "evaluation/eval_escalation.py"],
    text=True,
    capture_output=True
)

print(completed.stdout)

if completed.stderr:
    print("\n--- STDERR ---")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        "The evaluation script failed. Read the final lines above for the "
        "underlying error. Parts 13 and 14 will only work after this cell succeeds."
    )

print("\nEvaluation completed successfully.")


Using Colab cache for faster access to the 'fake-news-classification' dataset.
Loaded 9,398 held-out WELFake test articles


--- STDERR ---

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5080.19it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were 

RuntimeError: The evaluation script failed. Read the final lines above for the underlying error. Parts 13 and 14 will only work after this cell succeeds.

## Part 13 — Find and display the generated results

In [14]:
from pathlib import Path
import pandas as pd
from IPython.display import display

all_csvs = list(project_root.rglob("*.csv"))
all_jsons = list(project_root.rglob("*.json"))

print("CSV files found:")
for path in all_csvs:
    print(" -", path)

result_candidates = [
    path for path in all_csvs
    if any(word in path.name.lower() for word in ["escalation", "result", "verification"])
]

# Avoid accidentally choosing a dataset CSV where possible.
result_candidates = [
    path for path in result_candidates
    if "dataset" not in str(path).lower()
]

if not result_candidates:
    raise FileNotFoundError(
        "No generated results CSV was found. Part 12 did not create an output file."
    )

result_candidates.sort(
    key=lambda p: (
        "escalation_results" not in p.name.lower(),
        -p.stat().st_mtime
    )
)

results_path = result_candidates[0]
print("\nUsing results file:", results_path)

df = pd.read_csv(results_path)
display(df.head(20))

print("\nRows:", len(df))
print("Columns:", df.columns.tolist())

if "verification_source" in df.columns:
    print("\nVerification sources:")
    print(df["verification_source"].value_counts(dropna=False))

if "verdict" in df.columns:
    print("\nVerdicts:")
    print(df["verdict"].value_counts(dropna=False))

if {"classifier_label", "true_label"}.issubset(df.columns):
    classifier_accuracy = (
        df["classifier_label"].astype(str)
        == df["true_label"].astype(str)
    ).mean()
    print("\nClassifier-only accuracy:", classifier_accuracy)

if {"final_label", "true_label"}.issubset(df.columns):
    final_accuracy = (
        df["final_label"].astype(str)
        == df["true_label"].astype(str)
    ).mean()
    print("Final accuracy:", final_accuracy)

summary_candidates = [
    path for path in all_jsons
    if "summary" in path.name.lower() or "escalation" in path.name.lower()
]

summary_path = summary_candidates[0] if summary_candidates else None

if summary_path:
    print("\nSummary file:", summary_path)
    print(summary_path.read_text())
else:
    print("\nNo summary JSON was generated.")


CSV files found:
 - /content/Fake_news/evaluation/escalation_results.csv

Using results file: /content/Fake_news/evaluation/escalation_results.csv


,text_snippet,true_label,classifier_label,verdict,final_label
0,Barack Obama says memory of Hiroshima 'must ne...,real,real,supported,real
1,A Message to my Fellow Republicans. As the unf...,real,fake,supported,real
2,Pantsuit Power flashmob video for Hillary Clin...,real,real,supported,real
3,"Fox News built a f**ked-up Frankenstein, dumb,...",real,real,supported,real
4,House GOP smells victory in budget battle. “I ...,real,real,supported,real
5,Kevin McCarthy is a total dope: This bumbling ...,real,real,refuted,fake
6,Anti-Semitism growing in Europe. While I under...,real,real,supported,real
7,Kim Davis's right to religious liberty has bee...,real,real,insufficient,real
8,Donald Trump revokes Washington Post press acc...,real,fake,refuted,fake
9,"This man wants to become president, pass one l...",real,fake,supported,real



Rows: 100
Columns: ['text_snippet', 'true_label', 'classifier_label', 'verdict', 'final_label']

Verdicts:
verdict
supported       42
refuted         32
insufficient    26
Name: count, dtype: int64

Classifier-only accuracy: 0.76
Final accuracy: 0.64

No summary JSON was generated.


## Part 14 — Download the result files

In [15]:
from google.colab import files

print("Downloading:", results_path)
files.download(str(results_path))

if summary_path and summary_path.exists():
    print("Downloading:", summary_path)
    files.download(str(summary_path))
else:
    print("No summary JSON is available to download.")


Downloading: /content/Fake_news/evaluation/escalation_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

No summary JSON is available to download.


## Troubleshooting

If Part 7 or Part 9 says `classifier_head.pt` is missing, that trained file is not available in the cloned repository at the expected path. Upload the original trained file in Part 8.

If Part 12 fails, the notebook now shows the complete standard output and error output from `eval_escalation.py`, rather than allowing Parts 13–14 to fail with a misleading missing-CSV error.
